In [15]:
# ----------------------------
#  1. Load and inspect the data
# ----------------------------

import pandas as pd
import numpy as np

df = pd.read_csv('../Data/Raw/sales_raw.csv')

# Inspect the dataset
df.info()
print(f"\nRows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14060 entries, 0 to 14059
Data columns (total 23 columns):
 #   Column                                                                                     Non-Null Count  Dtype  
---  ------                                                                                     --------------  -----  
 0   STRUCTURE                                                                                  14060 non-null  object 
 1   STRUCTURE_ID                                                                               14060 non-null  object 
 2   STRUCTURE_NAME                                                                             14060 non-null  object 
 3   freq                                                                                       14060 non-null  object 
 4   Time frequency                                                                             14060 non-null  object 
 5   indic_bt                                      

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,indic_bt,Business trend indicator,nace_r2,Statistical classification of economic activities in the European Community (NACE Rev. 2),s_adj,...,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:STS_TRTU_A(1.0),Turnover and volume of sales in wholesale and ...,A,Annual,NETTUR,Net turnover,G47,"Retail trade, except of motor vehicles and mot...",CA,...,AT,Austria,2000,NaN,81.9,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:STS_TRTU_A(1.0),Turnover and volume of sales in wholesale and ...,A,Annual,NETTUR,Net turnover,G47,"Retail trade, except of motor vehicles and mot...",CA,...,AT,Austria,2001,NaN,82.0,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:STS_TRTU_A(1.0),Turnover and volume of sales in wholesale and ...,A,Annual,NETTUR,Net turnover,G47,"Retail trade, except of motor vehicles and mot...",CA,...,AT,Austria,2002,NaN,82.5,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:STS_TRTU_A(1.0),Turnover and volume of sales in wholesale and ...,A,Annual,NETTUR,Net turnover,G47,"Retail trade, except of motor vehicles and mot...",CA,...,AT,Austria,2003,NaN,83.5,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:STS_TRTU_A(1.0),Turnover and volume of sales in wholesale and ...,A,Annual,NETTUR,Net turnover,G47,"Retail trade, except of motor vehicles and mot...",CA,...,AT,Austria,2004,NaN,84.9,NaN,NaN,NaN,NaN,NaN


In [16]:
# =======================
# 2. filter indicators, drop and rename columns
# =======================


# filter indicators
df = df[df["nace_r2"].isin(["G47", "G47_NF_HLTH", "G476"])]

# keep only the needed columns ["geo", "TIME_PERIOD", "OBS_VALUE", "unit", "nace_r2"]
df = df[["geo", "TIME_PERIOD", "OBS_VALUE", "unit", "nace_r2"]]

df = df.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "sales_value",
    "unit": "sales_unit",
    "nace_r2": "indicator"
})

# show all unique indicators
print(df["indicator"].unique())

# check data types
print(df.dtypes)
print(df.head())


['G47' 'G476' 'G47_NF_HLTH']
country         object
year             int64
sales_value    float64
sales_unit      object
indicator       object
dtype: object
  country  year  sales_value sales_unit indicator
0      AT  2000         81.9        I10       G47
1      AT  2001         82.0        I10       G47
2      AT  2002         82.5        I10       G47
3      AT  2003         83.5        I10       G47
4      AT  2004         84.9        I10       G47


In [17]:
# ========================
# 3. clean data
# ========================
# ensure numeric
df["sales_value"] = pd.to_numeric(df["sales_value"], errors="coerce")

# remove invalid values only
df = df[df["sales_value"] > 0]



In [18]:
# ==========================
# 4. Aggregate and sort data
# ==========================

df = df.groupby(
    ["country", "year", "indicator"],
    as_index=False
)["sales_value"].mean()


df = df.sort_values(["country", "indicator", "year"])


In [19]:
# =====================
# 5. export the data
# =====================

# Export cleaned dataset
df.to_csv('../Data/Processed/sales_clean.csv', index=False)


In [20]:
# =============================
# 6. Validate output
# =============================

# check unique countries
print("Unique countries:", df["country"].nunique())

# check unique years
print("Unique years:", df["year"].nunique())

# check unique indicators (VERY important for sales)
print("Unique indicators:", df["indicator"].nunique())
print("Indicators:", df["indicator"].unique())

# total rows
print("Total rows:", len(df))

# year range
print("Year range:", df["year"].min(), "-", df["year"].max())

# sales value range
print("Sales value range:", df["sales_value"].min(), "-", df["sales_value"].max())

Unique countries: 41
Unique years: 35
Unique indicators: 3
Indicators: ['G47' 'G476' 'G47_NF_HLTH']
Total rows: 2186
Year range: 1991 - 2025
Sales value range: 7.383333333333333 - 330.96666666666664
